In [12]:
# ============================================================
# ЯЧЕЙКА 1: Импорты и исходные данные
# ============================================================
import json
import csv
import os
from collections import Counter

# Схемы таблиц
Commission = dict(id=int, name=str, code=str, specialization=int)
Specialization = dict(id=int, code=str)
SpecializationInCommission = dict(id=int, commission=int, specialization=int)
Scientist = dict(id=int, name=str, commission=int, specialization=int)
Publication = dict(id=int, title=str, scientist=int, journal=int)
Journal = dict(id=int, title=str)
JournalSpecialization = dict(journal=int, specialization=int)
ScientistInCommission = dict(scientist=int, commission=int, specialization=int)

# Таблицы данных
CommissionTable = [
    (1, "Совет по математике", "МАТ-01", 101),
    (2, "Совет по физике", "ФИЗ-02", 102),
    (3, "Совет по информатике", "ИНФ-03", 103),
]

SpecializationTable = [
    (101, "01.01.01"),
    (102, "01.04.02"),
    (103, "05.13.11"),
]

SpecializationInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

ScientistTable = [
    (1, "Иванов И.И.", 1, 101),
    (2, "Петров П.П.", 1, 102),
    (3, "Сидоров С.С.", 2, 102),
    (4, "Кузнецова А.А.", 3, 103),
]

PublicationTable = [
    (1, "Математические модели", 1, 10),
    (2, "Квантовая физика", 2, 11),
    (3, "Искусственный интеллект", 4, 12),
    (4, "Численные методы", 1, 13),
]

JournalTable = [
    (10, "Вестник РАН"),
    (11, "ЖЭТФ"),
    (12, "AI Journal"),
    (13, "Выч. математика"),
]

JournalSpecializationTable = [
    (10, 101),
    (11, 102),
    (12, 103),
    (13, 101),
]

ScientistInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

# Множество источников S
S = [
    ("Commission", Commission, CommissionTable),
    ("Specialization", Specialization, SpecializationTable),
    ("SpecializationInCommission", SpecializationInCommission, SpecializationInCommissionTable),
    ("Scientist", Scientist, ScientistTable),
    ("Publication", Publication, PublicationTable),
    ("Journal", Journal, JournalTable),
    ("JournalSpecialization", JournalSpecialization, JournalSpecializationTable),
    ("ScientistInCommission", ScientistInCommission, ScientistInCommissionTable),
]

In [13]:
# ЯЧЕЙКА 2: Сохранение правил в JSON (с entity_resolution)
import json

rules_json = {
    "ontology": {
        "name": "Онтология академической среды",
        "classes": ["Учёный", "Публикация", "Журнал", "Специальность", "Диссовет"],
        "relations": [
            "фио", "название", "шифр", "код",
            "опубликовал_статью", "опубликована_в",
            "покрывает_специальность", "включает_специальность",
            "состоит_в_совете", "представляет_специальность"
        ]
    },

    "entity_resolution": {
        "mappings": [
            {"source_field": "Scientist.id", "target_entity": "Учёный", "is_primary_key": True},
            {"source_field": "Publication.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
            {"source_field": "Publication.id", "target_entity": "Публикация", "is_primary_key": True},
            {"source_field": "Publication.journal", "target_entity": "Журнал", "references": "Journal.id"},
            {"source_field": "Journal.id", "target_entity": "Журнал", "is_primary_key": True},
            {"source_field": "JournalSpecialization.journal", "target_entity": "Журнал", "references": "Journal.id"},
            {"source_field": "JournalSpecialization.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
            {"source_field": "Specialization.id", "target_entity": "Специальность", "is_primary_key": True},
            {"source_field": "Commission.id", "target_entity": "Диссовет", "is_primary_key": True},
            {"source_field": "SpecializationInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
            {"source_field": "SpecializationInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
            {"source_field": "ScientistInCommission.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
            {"source_field": "ScientistInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
            {"source_field": "ScientistInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"}
        ]
    },

    "rules": {
        "structured": [
            {"source": "Scientist", "subject_field": "id", "subject_entity": "Учёный", "relation": "фио", "object_field": "name", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "id", "subject_entity": "Публикация", "relation": "название", "object_field": "title", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "опубликовал_статью", "object_field": "id", "object_entity": "Публикация", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "id", "subject_entity": "Публикация", "relation": "опубликована_в", "object_field": "journal", "object_entity": "Журнал", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Journal", "subject_field": "id", "subject_entity": "Журнал", "relation": "название", "object_field": "title", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "JournalSpecialization", "subject_field": "journal", "subject_entity": "Журнал", "relation": "покрывает_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Commission", "subject_field": "id", "subject_entity": "Диссовет", "relation": "название", "object_field": "name", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Commission", "subject_field": "id", "subject_entity": "Диссовет", "relation": "шифр", "object_field": "code", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Specialization", "subject_field": "id", "subject_entity": "Специальность", "relation": "код", "object_field": "code", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "SpecializationInCommission", "subject_field": "commission", "subject_entity": "Диссовет", "relation": "включает_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "ScientistInCommission", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "состоит_в_совете", "object_field": "commission", "object_entity": "Диссовет", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "ScientistInCommission", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "представляет_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0}
        ],
        "semi_structured": [],
        "unstructured": []
    }
}

with open("ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)
print("✓ ontology_rules.json сохранён")

✓ ontology_rules.json сохранён


In [14]:
# ЯЧЕЙКА 3: Функции интеграции с entity_resolution
def resolve_entity_id(source_name, field_name, field_value, entity_type, entity_resolution):
    """Формирует глобальный ID сущности по карте ссылочной целостности."""
    full_field = f"{source_name}.{field_name}"
    
    # Ищем в карте
    for mapping in entity_resolution.get("mappings", []):
        if mapping["source_field"] == full_field:
            if mapping.get("is_primary_key"):
                return f"{source_name}_{field_value}"
            elif "references" in mapping:
                target_source, _ = mapping["references"].rsplit(".", 1)
                return f"{target_source}_{field_value}"
    
    # Если нет в карте — возвращаем как есть
    return str(field_value)


def apply_rule(source_name, rule, data_sources, entity_resolution):
    """Применяет одно правило с учётом ссылочной целостности."""
    facts = []
    if source_name not in data_sources:
        return facts
    
    schema, table = data_sources[source_name]
    columns = list(schema.keys())
    
    for row in table:
        row_dict = dict(zip(columns, row))
        
        # Субъект
        subj_field = rule["subject_field"]
        subj_entity = rule.get("subject_entity")
        subject = resolve_entity_id(source_name, subj_field, row_dict[subj_field], subj_entity, entity_resolution)
        
        # Объект
        if "object_value" in rule:
            obj = rule["object_value"]
        else:
            obj_field = rule["object_field"]
            obj_entity = rule.get("object_entity")
            obj = resolve_entity_id(source_name, obj_field, row_dict[obj_field], obj_entity, entity_resolution)
        
        # Время
        if "time_field" in rule and rule["time_field"] in row_dict:
            time_val = str(row_dict[rule["time_field"]])
        else:
            time_val = rule.get("time_default", "2025-01-01")
        
        # Значение
        value = row_dict[rule["value_field"]] if "value_field" in rule and rule["value_field"] in row_dict else None
        
        facts.append((subject, rule["relation"], obj, time_val, value, rule.get("confidence", 1.0)))
    
    return facts


def integrate(S, rules_file="ontology_rules.json"):
    """Интеграция всех источников по правилам."""
    with open(rules_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    ontology = config["ontology"]
    rules = config["rules"]
    entity_resolution = config.get("entity_resolution", {"mappings": []})
    
    data_sources = {name: (schema, table) for name, schema, table in S}
    all_facts, stats = [], Counter()
    
    for rule_type in ["structured", "semi_structured", "unstructured"]:
        for rule in rules.get(rule_type, []):
            facts = apply_rule(rule["source"], rule, data_sources, entity_resolution)
            all_facts.extend(facts)
            stats[rule["relation"]] += len(facts)
    
    return all_facts, stats, ontology

print("✓ integrate и apply_rule загружены")

✓ integrate и apply_rule загружены


In [15]:
# ============================================================
# ЯЧЕЙКА 4: Запуск интеграции
# ============================================================
import os

print("=" * 60)
print("ОНТОЛОГИЧЕСКАЯ ИНТЕГРАЦИЯ ДАННЫХ О ДИССОВЕТАХ")
print("=" * 60)

# Проверяем S
print(f"\nИсточников данных: {len(S)}")
for name, schema, table in S:
    print(f"  {name}: {len(table)} записей (колонки: {', '.join(schema.keys())})")

# Запуск интеграции
facts, stats, ontology = integrate(S)

print(f"\nВсего фактов: {len(facts)}")
print(f"\nРаспределение по отношениям:")
for rel, count in sorted(stats.items()):
    print(f"  {rel}: {count}")

# Проверка покрытия
expected = set(ontology["relations"])
actual = set(stats.keys())
if expected - actual:
    print(f"\nВНИМАНИЕ: Отсутствуют отношения: {expected - actual}")
else:
    print(f"\n✓ Покрытие онтологии: 100% ({len(actual)}/{len(expected)} отношений)")

# В конец ячейки 4 добавить:
output_path = "experiment_data_soviets/generated/all_facts.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['subject', 'relation', 'object', 'time', 'n_value', 'confidence'])
    for fact in facts:
        writer.writerow(fact)
print(f"✓ Сохранено {len(facts)} фактов в {output_path}")

# Вывод первых 5 фактов
print(f"\nПримеры фактов:")
for fact in facts[:5]:
    print(f"  {fact}")

print("\nГотово!")

ОНТОЛОГИЧЕСКАЯ ИНТЕГРАЦИЯ ДАННЫХ О ДИССОВЕТАХ

Источников данных: 8
  Commission: 3 записей (колонки: id, name, code, specialization)
  Specialization: 3 записей (колонки: id, code)
  SpecializationInCommission: 4 записей (колонки: id, commission, specialization)
  Scientist: 4 записей (колонки: id, name, commission, specialization)
  Publication: 4 записей (колонки: id, title, scientist, journal)
  Journal: 4 записей (колонки: id, title)
  JournalSpecialization: 4 записей (колонки: journal, specialization)
  ScientistInCommission: 4 записей (колонки: scientist, commission, specialization)

Всего фактов: 45

Распределение по отношениям:
  включает_специальность: 4
  код: 3
  название: 11
  опубликовал_статью: 4
  опубликована_в: 4
  покрывает_специальность: 4
  представляет_специальность: 4
  состоит_в_совете: 4
  фио: 4
  шифр: 3

✓ Покрытие онтологии: 100% (10/10 отношений)
✓ Сохранено 45 фактов в experiment_data_soviets/generated/all_facts.csv

Примеры фактов:
  ('Scientist_1', 'фио

In [46]:
# ЯЧЕЙКА: Сохранение всех файлов проекта
import json
import csv
import os

# Создаём папку проекта
project_dir = "dissovet_ontology"
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/data", exist_ok=True)

# Сохраняем ячейки как Python-скрипт
with open(f"{project_dir}/module1_integrator.py", 'w', encoding='utf-8') as f:
    f.write("""# Здесь ваш код из ячейки 3 (функции integrate, apply_rule, load_rules, save_to_csv)
""")

# Сохраняем правила
with open(f"{project_dir}/ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)

# Сохраняем CSV-файлы
for name, schema, table in S:
    columns = list(schema.keys())
    with open(f"{project_dir}/data/{name}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(table)

# Сохраняем README.md
with open(f"{project_dir}/README.md", 'w', encoding='utf-8') as f:
    f.write("""# Онтологическая интеграция данных диссертационных советов

Проект выполняет интеграцию гетерогенных данных в унифицированные спецификации фактов (УСФ).

## Структура

- `ontology_rules.json` — правила отображения
- `data/*.csv` — исходные таблицы
- `module1_integrator.py` — модуль интеграции

## Запуск

Скопируйте функции из `module1_integrator.py` в ноутбук.
""")

# Сохраняем .gitignore
with open(f"{project_dir}/.gitignore", 'w') as f:
    f.write("""__pycache__/
*.pyc
.ipynb_checkpoints/
*.db
experiment_data*/
""")

print("✓ Файлы сохранены в dissovet_ontology/")

✓ Файлы сохранены в dissovet_ontology/
